In [11]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

df = pd.read_csv("All_MDD_LLM.csv")

# feature and target columns
feature_cols = df.columns[1:38]
# feature_cols = df.columns[1:38].tolist() + df.columns[47:].tolist()
# feature_cols = df.columns[1:25].tolist() + [df.columns[37]]
target_cols = df.columns[38:47]
print(feature_cols)

Index(['P/E mean', 'P/E standard deviation', 'P/B mean',
       'P/B standard deviation', 'P/S mean', 'P/S standard deviation',
       'P/CF mean', 'P/CF standard deviation', 'EV/EBITDA mean',
       'EV/EBITDA standard deviation', 'Gross Margin mean',
       'Gross Margin standard deviation', 'Net Margin mean',
       'Net Margin standard deviation', 'ROA mean', 'ROA standard deviation',
       'ROE mean', 'ROE standard deviation', 'FCF mean',
       'FCF standard deviation', 'FCF per Share mean',
       'FCF per Share standard deviation', 'SPS mean',
       'SPS standard deviation', 'PVS mean', 'PVS standard deviation',
       'RAPS mean', 'RAPS standard deviation', 'EVC mean',
       'EVC standard deviation', 'VEC mean', 'VEC standard deviation',
       'PLF mean', 'PLF standard deviation', 'IQS mean',
       'IQS standard deviation', 'Period'],
      dtype='object')


In [12]:
X = df[feature_cols]
instances = df["instance"]

# 保存每个算法的预测和实际
all_actual = pd.DataFrame(index=df.index, columns=target_cols)
all_pred = pd.DataFrame(index=df.index, columns=target_cols)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for target in target_cols:
    y = df[target]

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 用 Pipeline 确保 MLP 收敛（加 scaler）
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(hidden_layer_sizes=(128, 64),
                                 activation='relu',
                                 solver='adam',
                                 learning_rate_init=0.001,
                                 max_iter=500,
                                 random_state=42))
        ])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 存储结果
        all_actual.loc[y_test.index, target] = y_test
        all_pred.loc[y_test.index, target] = y_pred

# === 计算结果 ===
result_summary = pd.DataFrame(index=df.index)
result_summary["Instance"] = instances

# SBS: OH-L 实际值
result_summary["SBS"] = all_actual["MS-T"]

# VBS: 所有算法中实际分数的最大值
result_summary["VBS"] = all_actual.max(axis=1)

# Selection Result: 根据预测分数选择算法，然后取该算法的 actual
best_pred_algo = all_pred.idxmax(axis=1)  # 每个样本预测最高的算法名
result_summary["Selected Algo"] = best_pred_algo

selection_result = []
for i, algo in best_pred_algo.items():
    selection_result.append(all_actual.loc[i, algo])
result_summary["Selection Result"] = selection_result

# === 统计结果 ===
stats = pd.DataFrame({
    "Mean": result_summary[["SBS", "VBS", "Selection Result"]].mean(),
    "Std": result_summary[["SBS", "VBS", "Selection Result"]].std(),
    "Min": result_summary[["SBS", "VBS", "Selection Result"]].min(),
    "Max": result_summary[["SBS", "VBS", "Selection Result"]].max(),
}).T

# === 统计各算法被选中的次数 ===
algo_counts = result_summary["Selected Algo"].value_counts().rename_axis("Algorithm").reset_index(name="Count")
algo_counts["Proportion"] = algo_counts["Count"] / len(result_summary)

print("=== 每个样本的结果（前几行） ===")
print(result_summary.head())

print("\n=== 总体统计 ===")
print(stats)

print("\n=== 算法被选中次数 ===")
print(algo_counts)

# 保存
result_summary.to_csv("MDD_strategy_selection_MLP.csv", index=False)
stats.to_csv("MDD_strategy_selection_stats_MLP.csv")
algo_counts.to_csv("MDD_strategy_selection_algo_counts_MLP.csv", index=False)


=== 每个样本的结果（前几行） ===
   Instance       SBS       VBS Selected Algo  Selection Result
0         1 -0.154842 -0.121491          MS-L         -0.150075
1         2 -0.154842 -0.121491          MS-L         -0.150075
2         3 -0.250673 -0.195531          MS-M         -0.248521
3         4 -0.250673 -0.237388          MS-L         -0.237388
4         5 -0.250673 -0.237388          MS-T         -0.250673

=== 总体统计 ===
           SBS       VBS Selection Result
Mean -0.277681 -0.262082        -0.274834
Std   0.093319  0.092745         0.092814
Min  -0.546318 -0.528406         -0.60195
Max  -0.060414 -0.059724        -0.061189

=== 算法被选中次数 ===
  Algorithm  Count  Proportion
0        EW    292    0.383202
1      MS-T    187    0.245407
2      MS-L    103    0.135171
3       MVO     61    0.080052
4      MS-M     44    0.057743
5       MCW     44    0.057743
6      OH-T     17    0.022310
7      OH-L      7    0.009186
8      OH-M      7    0.009186
